In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q qdrant-client weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.7/652.7 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.9 MB/s eta 0:00:00


In [2]:
import os
import re
import torch
import time
import gc
import pandas as pd
import random
import numpy as np
import json
import pickle
from tqdm import tqdm
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

# jinaa5-nano

# Load chunks and embeddings

In [4]:
# ===================== CONFIG =====================
save_folder = "/content/drive/MyDrive/Book_Embeddings_100/jinaa5_nano"
model_name  = "jinaai/jina-embeddings-v5-text-nano-retrieval"

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== RELOAD ALL SAVED EMBEDDINGS =====================
all_chunks_text = []
all_embeddings  = []
all_metadata    = []

pkl_files = [f for f in os.listdir(save_folder) if f.endswith('.pkl')]
print(f"Found {len(pkl_files)} embedding files")

for pkl_file in tqdm(pkl_files, desc="Loading embeddings"):
    file_path = os.path.join(save_folder, pkl_file)
    with open(file_path, "rb") as f:
        data = pickle.load(f)

    chunks     = data["chunks"]
    embeddings = data["embeddings"]
    book_title = data["book_title"]
    file_name  = data["file_name"]

    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        all_chunks_text.append(chunk)
        all_embeddings.append(emb)
        all_metadata.append({
            "file_name": file_name,
            "book_title": book_title,
            "chunk_id": i
        })

all_embeddings = np.array(all_embeddings).astype('float32')

print(f"""
============== LOADED DATA (jina-v5-nano) ==============
  Total books:      {len(pkl_files)}
  Total chunks:     {len(all_chunks_text)}
  Embedding dim:    {all_embeddings.shape[1]}
  Matrix size:      {all_embeddings.nbytes/1024**2:.2f} MB
=========================================================
""")

Loading model on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.22k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

Model loaded!

Found 100 embedding files


Loading embeddings: 100%|██████████| 100/100 [01:41<00:00,  1.02s/it]



============== LOADED DATA (jina-v5-nano) ==============
  Total books:      100
  Total chunks:     88275
  Embedding dim:    768
  Matrix size:      258.62 MB



# Load model for query encoding

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

model_name = "jinaai/jina-embeddings-v5-text-nano-retrieval"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded for query encoding!")

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

# Build Qdrant + Weaviate indexes

In [6]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

dim = all_embeddings.shape[1]


# ===================== RECREATE WEAVIATE COLLECTION WITH chunk_id =====================
# ---- Weaviate ----
client_weaviate = weaviate.connect_to_embedded()
weaviate_collection = "BooksJina5nano"

if client_weaviate.collections.exists(weaviate_collection):
    client_weaviate.collections.delete(weaviate_collection)

client_weaviate.collections.create(
    name=weaviate_collection,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[
        Property(name="file_name", data_type=DataType.TEXT),
        Property(name="chunk_id",  data_type=DataType.INT)
    ]
)

print("Re-indexing into Weaviate...")
weaviate_index_start = time.time()
col = client_weaviate.collections.get(weaviate_collection)
with col.batch.fixed_size(batch_size=512) as batch:
    for i in tqdm(range(len(all_embeddings)), desc="Weaviate upload"):
        batch.add_object(
            properties={
                "file_name": all_metadata[i]["file_name"],
                "chunk_id":  all_metadata[i]["chunk_id"]
            },
            vector=all_embeddings[i].tolist()
        )
weaviate_index_time = time.time() - weaviate_index_start
print(f"Weaviate indexing time: {weaviate_index_time:.2f}s\n")


# ---- Qdrant ----
client_qdrant = QdrantClient(":memory:")
qdrant_collection = "books_jina5nano"

client_qdrant.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
)

print("Re-indexing into Qdrant...")
qdrant_index_start = time.time()
for i in tqdm(range(0, len(all_embeddings), 512), desc="Qdrant upload"):
    batch_end = min(i + 512, len(all_embeddings))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings[j].tolist(),
            payload={
                "file_name": all_metadata[j]["file_name"],
                "chunk_id": all_metadata[j]["chunk_id"]
            }
        )
        for j in range(i, batch_end)
    ]
    client_qdrant.upsert(collection_name=qdrant_collection, points=points)
qdrant_index_time = time.time() - qdrant_index_start
print(f"Qdrant indexing time: {qdrant_index_time:.2f}s\n")


# ---- Search functions ----
def qdrant_search_detailed(query_emb, k=50):
    results = client_qdrant.query_points(
        collection_name=qdrant_collection, query=query_emb, limit=k, with_payload=True
    ).points
    return [(r.payload["file_name"], r.payload.get("chunk_id")) for r in results]

def weaviate_search_detailed(query_emb, k=50):
    results = col.query.near_vector(near_vector=query_emb, limit=k, return_properties=["file_name", "chunk_id"])
    return [(r.properties["file_name"], r.properties.get("chunk_id")) for r in results.objects]

INFO:weaviate-client:Binary /root/.cache/weaviate-embedded did not exist. Downloading binary from https://github.com/weaviate/weaviate/releases/download/v1.30.5/weaviate-v1.30.5-Linux-amd64.tar.gz
INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 2199


Re-indexing into Weaviate...


Weaviate upload: 100%|██████████| 88275/88275 [02:15<00:00, 653.43it/s]


Weaviate indexing time: 138.65s

Re-indexing into Qdrant...


Qdrant upload:  23%|██▎       | 39/173 [00:15<01:23,  1.60it/s]/tmp/ipykernel_901/1961477916.py:66: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client_qdrant.upsert(collection_name=qdrant_collection, points=points)
Qdrant upload: 100%|██████████| 173/173 [01:13<00:00,  2.36it/s]

Qdrant indexing time: 73.38s



# Evaluation functions

In [7]:
# ===================== UPDATED SEARCH FUNCTIONS (return chunk-level info) =====================

def qdrant_search_detailed(query_emb, k=50):
    """Returns raw chunk-level results: list of (file_name, chunk_id) tuples in rank order"""
    results = client_qdrant.query_points(
        collection_name=qdrant_collection,
        query=query_emb,
        limit=k,
        with_payload=True
    ).points

    return [(r.payload["file_name"], r.payload.get("chunk_id")) for r in results]


def weaviate_search_detailed(query_emb, k=50):
    """Returns raw chunk-level results: list of (file_name, chunk_id) tuples in rank order"""
    results = col.query.near_vector(
        near_vector=query_emb,
        limit=k,
        return_properties=["file_name", "chunk_id"]
    )

    return [(r.properties["file_name"], r.properties.get("chunk_id")) for r in results.objects]


def evaluate_robustness_full(search_fn_detailed, name, benchmark_df, deletion_pct):
    doc_hits_at_1, doc_hits_at_5, doc_hits_at_10 = 0, 0, 0
    chunk_hits_at_1, chunk_hits_at_5, chunk_hits_at_10 = 0, 0, 0
    latencies = []

    for _, row in tqdm(benchmark_df.iterrows(), desc=f"Evaluating {name} ({deletion_pct}%)", total=len(benchmark_df)):
        query = row["noisy_query"]
        expected_file  = row["expected_file"]
        expected_chunk = row["expected_chunk_id"]

        query_emb = model.encode([query], normalize_embeddings=True)[0].tolist()

        t0 = time.time()
        raw_results = search_fn_detailed(query_emb, k=50)
        latency_ms = (time.time() - t0) * 1000
        latencies.append(latency_ms)

        # Chunk-level
        chunk_matches = [
            (fname == expected_file and cid == expected_chunk)
            for fname, cid in raw_results
        ]
        if any(chunk_matches[:1]):
            chunk_hits_at_1 += 1
        if any(chunk_matches[:5]):
            chunk_hits_at_5 += 1
        if any(chunk_matches[:10]):
            chunk_hits_at_10 += 1

        # Doc-level
        retrieved_files, seen = [], set()
        for fname, _ in raw_results:
            if fname not in seen:
                seen.add(fname)
                retrieved_files.append(fname)
            if len(retrieved_files) == 10:
                break

        if retrieved_files and retrieved_files[0] == expected_file:
            doc_hits_at_1 += 1
        if expected_file in retrieved_files[:5]:
            doc_hits_at_5 += 1
        if expected_file in retrieved_files[:10]:
            doc_hits_at_10 += 1

    n = len(benchmark_df)
    lat = np.array(latencies)

    return {
        "vector_store": name,
        "model": "jina-v5-nano",
        "test_type": f"{deletion_pct}%_token_deletion",
        "doc_hit@1":  round(doc_hits_at_1 / n, 4),
        "doc_hit@5":  round(doc_hits_at_5 / n, 4),
        "doc_hit@10": round(doc_hits_at_10 / n, 4),
        "chunk_hit@1":  round(chunk_hits_at_1 / n, 4),
        "chunk_hit@5":  round(chunk_hits_at_5 / n, 4),
        "chunk_hit@10": round(chunk_hits_at_10 / n, 4),
        "latency_mean_ms": round(lat.mean(), 2),
        "latency_p95_ms":  round(np.percentile(lat, 95), 2),
    }


# Create test benchmark with n% replacement rate

In [8]:
import random
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

def create_replacement_benchmark(all_chunks_text, all_metadata, all_embeddings=None,
                                 n_samples=1000, replacement_ratio=0.20,
                                 vocab_sample_size=5000, random_seed=42):
    # Set random seeds
    random.seed(random_seed)
    np.random.seed(random_seed)

    # Sample random chunks
    total_chunks = len(all_chunks_text)
    sample_indices = random.sample(range(total_chunks), min(n_samples, total_chunks))

    sampled_chunks = [all_chunks_text[i] for i in sample_indices]
    sampled_metadata = [all_metadata[i] for i in sample_indices]

    # Build vocabulary pool for random replacement
    print("Building vocabulary pool from corpus...")
    vocab_pool = []
    sample_size = min(vocab_sample_size, len(all_chunks_text))
    for chunk in tqdm(random.sample(all_chunks_text, sample_size), desc="Sampling vocab"):
        vocab_pool.extend(chunk.split())

    vocab_pool = list(set(vocab_pool))  # unique tokens
    print(f"Vocabulary pool size: {len(vocab_pool)} unique tokens")

    # Replace random tokens
    def replace_random_tokens(text, ratio, vocab):
        tokens = text.split()
        n_replace = int(len(tokens) * ratio)

        if n_replace == 0 or len(tokens) <= 1:
            return text

        replace_indices = set(random.sample(range(len(tokens)), n_replace))
        new_tokens = [
            random.choice(vocab) if i in replace_indices else t
            for i, t in enumerate(tokens)
        ]
        return " ".join(new_tokens)

    noisy_queries = [replace_random_tokens(chunk, replacement_ratio, vocab_pool)
                     for chunk in sampled_chunks]

    print(f"\n--- Example corruption ({replacement_ratio*100:.0f}% random replacement) ---")
    print(f"Original: {sampled_chunks[0][:200]}...")
    print(f"Noisy:    {noisy_queries[0][:200]}...")

    # Build benchmark DataFrame
    replacement_benchmark_df = pd.DataFrame({
        "id": range(len(sample_indices)),
        "original_chunk": sampled_chunks,
        "noisy_query": noisy_queries,
        "expected_file": [m["file_name"] for m in sampled_metadata],
        "expected_chunk_id": [m["chunk_id"] for m in sampled_metadata],
        "original_chunk_index": sample_indices
    })

    print(f"\nBuilt {replacement_ratio*100:.0f}% replacement benchmark with {len(replacement_benchmark_df)} queries")

    return replacement_benchmark_df




#10%

In [10]:
benchmark_df_10 = create_replacement_benchmark(
    all_chunks_text, all_metadata, all_embeddings,
    n_samples=1000,
    replacement_ratio=0.10,  # <-- Change this value
    random_seed=42
)
benchmark_df_10.to_csv("/content/drive/MyDrive/replacement_benchmark_10pct.csv", index=False)


qdrant_robustness_10   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_10, 10)
weaviate_robustness_10 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_10, 10)

robustness_10_df = pd.DataFrame([qdrant_robustness_10, weaviate_robustness_10])
print(robustness_10_df.to_string(index=False))

robustness_10_df.to_csv("/content/drive/MyDrive/replacement_results_10pct.csv", index=False)

Building vocabulary pool from corpus...


Sampling vocab:   0%|          | 0/5000 [00:00<?, ?it/s]

Vocabulary pool size: 65456 unique tokens

--- Example corruption (10% random replacement) ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز النِّسَاءِ، که عَشْرٍ این خطابات را داشته باشد، این (زخرف را نمی فرماید. عبارت دیگر قد اذنت لک است که ا...

Built 10% replacement benchmark with 1000 queries


Evaluating Qdrant (10%):   0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Evaluating Weaviate (10%):   0%|          | 0/1000 [00:00<?, ?it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 10%_token_deletion      0.851      0.978       0.982        0.834        0.963         0.963           398.34          484.37
    Weaviate jina-v5-nano 10%_token_deletion      0.852      0.971       0.977        0.838        0.961         0.962             9.25           11.99


# 20%

In [11]:
benchmark_df_20 = create_replacement_benchmark(
    all_chunks_text, all_metadata, all_embeddings,
    n_samples=1000,
    replacement_ratio=0.20,  # <-- Change this value
    random_seed=42
)
benchmark_df_20.to_csv("/content/drive/MyDrive/replacement_benchmark_20pct.csv", index=False)


qdrant_robustness_20   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_20, 20)
weaviate_robustness_20 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_20, 20)

robustness_20_df = pd.DataFrame([qdrant_robustness_20, weaviate_robustness_20])
print(robustness_20_df.to_string(index=False))

robustness_20_df.to_csv("/content/drive/MyDrive/replacement_results_20pct.csv", index=False)

Building vocabulary pool from corpus...


Sampling vocab:   0%|          | 0/5000 [00:00<?, ?it/s]

Vocabulary pool size: 65456 unique tokens

--- Example corruption (20% random replacement) ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه وَلِیّاً رحمت و فیض عوالم وجوی است بهفردی جز الشَّكُ؛ که أَحَبَّكَ این خطابات را داشته باشد، این برزخی را نمی فرماید. عبارت دیگر قد اذنت لک ا...

Built 20% replacement benchmark with 1000 queries


Evaluating Qdrant (20%):   0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Evaluating Weaviate (20%):   0%|          | 0/1000 [00:00<?, ?it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 20%_token_deletion      0.852      0.970       0.979        0.832        0.952         0.956           446.92          780.85
    Weaviate jina-v5-nano 20%_token_deletion      0.830      0.959       0.971        0.811        0.944         0.950             9.46           12.33


# 30%

In [12]:
benchmark_df_30 = create_replacement_benchmark(
    all_chunks_text, all_metadata, all_embeddings,
    n_samples=1000,
    replacement_ratio=0.30,  # <-- Change this value
    random_seed=42
)
benchmark_df_30.to_csv("/content/drive/MyDrive/replacement_benchmark_30pct.csv", index=False)


qdrant_robustness_30   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_30, 30)
weaviate_robustness_30 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_30, 30)

robustness_30_df = pd.DataFrame([qdrant_robustness_30, weaviate_robustness_30])
print(robustness_30_df.to_string(index=False))

robustness_30_df.to_csv("/content/drive/MyDrive/replacement_results_30pct.csv", index=False)

Building vocabulary pool from corpus...


Sampling vocab:   0%|          | 0/5000 [00:00<?, ?it/s]

Vocabulary pool size: 65456 unique tokens

--- Example corruption (30% random replacement) ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه حَبیبِ رحمت و فیض عوالم وجوی است علاّمه سرافكند جمعی يحيي قنّعتها این خطابات را داشته باشد، این 88318722 را نمی فرماید. عبارت دیگر قد اذنت لک...

Built 30% replacement benchmark with 1000 queries


Evaluating Qdrant (30%):   0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Evaluating Weaviate (30%):   0%|          | 0/1000 [00:00<?, ?it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 30%_token_deletion      0.794      0.942       0.962        0.758        0.911         0.924           407.61          497.05
    Weaviate jina-v5-nano 30%_token_deletion      0.784      0.925       0.949        0.746        0.895         0.908            11.36           14.79


%40

In [13]:
benchmark_df_40 = create_replacement_benchmark(
    all_chunks_text, all_metadata, all_embeddings,
    n_samples=1000,
    replacement_ratio=0.40,  # <-- Change this value
    random_seed=42
)
benchmark_df_40.to_csv("/content/drive/MyDrive/replacement_benchmark_40pct.csv", index=False)


qdrant_robustness_40   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_40, 40)
weaviate_robustness_40 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_40, 40)

robustness_40_df = pd.DataFrame([qdrant_robustness_40, weaviate_robustness_40])
print(robustness_40_df.to_string(index=False))

robustness_40_df.to_csv("/content/drive/MyDrive/replacement_results_40pct.csv", index=False)

Building vocabulary pool from corpus...


Sampling vocab:   0%|          | 0/5000 [00:00<?, ?it/s]

Vocabulary pool size: 65456 unique tokens

--- Example corruption (40% random replacement) ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه نبرد رحمت 6؛ فیض عوالم وجوی است اَنْوَرَ مدلل کشید، ابْتَدَاهَا...» فَهُوَ این خطابات را داشته باشد، این أَلْوَى را نمی دشواریها عبارت بخاری،...

Built 40% replacement benchmark with 1000 queries


Evaluating Qdrant (40%):   0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Evaluating Weaviate (40%):   0%|          | 0/1000 [00:00<?, ?it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 40%_token_deletion      0.628      0.858       0.900        0.577        0.779         0.805           390.05          477.75
    Weaviate jina-v5-nano 40%_token_deletion      0.615      0.833       0.878        0.562        0.756         0.781            10.29           13.45


# 50 %

In [14]:
benchmark_df_50 = create_replacement_benchmark(
    all_chunks_text, all_metadata, all_embeddings,
    n_samples=1000,
    replacement_ratio=0.50,  # <-- Change this value
    random_seed=42
)
benchmark_df_50.to_csv("/content/drive/MyDrive/replacement_benchmark_50pct.csv", index=False)


qdrant_robustness_50   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_50, 50)
weaviate_robustness_50 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_50, 50)

robustness_50_df = pd.DataFrame([qdrant_robustness_50, weaviate_robustness_50])
print(robustness_50_df.to_string(index=False))

robustness_50_df.to_csv("/content/drive/MyDrive/replacement_results_50pct.csv", index=False)

Building vocabulary pool from corpus...


Sampling vocab:   0%|          | 0/5000 [00:00<?, ?it/s]

Vocabulary pool size: 65456 unique tokens

--- Example corruption (50% random replacement) ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه فَالْزَمْهُ رحمت معد أمضي عوالم وجوی است تَتَوَکَّفُونَ العبا ولدها وحِذاءً اکمل ندامت خطابات را واجد باشد، این ساعده را نمی اَلْأُمَّةِ ننمو...

Built 50% replacement benchmark with 1000 queries


Evaluating Qdrant (50%):   0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Evaluating Weaviate (50%):   0%|          | 0/1000 [00:00<?, ?it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 50%_token_deletion      0.439      0.685       0.779        0.363        0.551         0.605           391.07          481.03
    Weaviate jina-v5-nano 50%_token_deletion      0.435      0.675       0.766        0.366        0.538         0.590            10.00           12.99


# jinaa3

In [15]:
!pip install -q \
  sentence-transformers==3.3.1 \
  transformers==4.47.1 \
  tokenizers==0.21.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.8 MB/s eta 0:00:00


# Load chunks and embeddings

In [3]:
# ===================== CONFIG =====================
save_folder = "/content/drive/MyDrive/Book_Embeddings_100/jinaa3"
model_name  = "jinaai/jina-embeddings-v3"

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== RELOAD ALL SAVED EMBEDDINGS =====================
all_chunks_text = []
all_embeddings  = []
all_metadata    = []

pkl_files = [f for f in os.listdir(save_folder) if f.endswith('.pkl')]
print(f"Found {len(pkl_files)} embedding files")

for pkl_file in tqdm(pkl_files, desc="Loading embeddings"):
    file_path = os.path.join(save_folder, pkl_file)
    with open(file_path, "rb") as f:
        data = pickle.load(f)

    chunks     = data["chunks"]
    embeddings = data["embeddings"]
    book_title = data["book_title"]
    file_name  = data["file_name"]

    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        all_chunks_text.append(chunk)
        all_embeddings.append(emb)
        all_metadata.append({
            "file_name": file_name,
            "book_title": book_title,
            "chunk_id": i
        })

all_embeddings = np.array(all_embeddings).astype('float32')

print(f"""
============== LOADED DATA (jina-v5-nano) ==============
  Total books:      {len(pkl_files)}
  Total chunks:     {len(all_chunks_text)}
  Embedding dim:    {all_embeddings.shape[1]}
  Matrix size:      {all_embeddings.nbytes/1024**2:.2f} MB
=========================================================
""")

Loading model on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/192 [00:00<?, ?B/s]

Model loaded!

Found 100 embedding files


Loading embeddings: 100%|██████████| 100/100 [01:47<00:00,  1.08s/it]



============== LOADED DATA (jina-v5-nano) ==============
  Total books:      100
  Total chunks:     88275
  Embedding dim:    1024
  Matrix size:      344.82 MB



# Load model for query encoding

In [4]:
import torch
from sentence_transformers import SentenceTransformer

model_name = "jinaai/jina-embeddings-v3"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded for query encoding!")

Model loaded for query encoding!


# Build Qdrant + Weaviate indexes for jina-v5-nano

In [5]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType

# ===================== RECREATE WEAVIATE COLLECTION WITH chunk_id =====================
# ---- Weaviate ----
client_weaviate = weaviate.connect_to_embedded()
weaviate_collection = "BooksJina3"

if client_weaviate.collections.exists(weaviate_collection):
    client_weaviate.collections.delete(weaviate_collection)

client_weaviate.collections.create(
    name=weaviate_collection,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[
        Property(name="file_name", data_type=DataType.TEXT),
        Property(name="chunk_id",  data_type=DataType.INT)
    ]
)

print("Re-indexing into Weaviate...")
weaviate_index_start = time.time()
col = client_weaviate.collections.get(weaviate_collection)
with col.batch.fixed_size(batch_size=512) as batch:
    for i in tqdm(range(len(all_embeddings)), desc="Weaviate upload"):
        batch.add_object(
            properties={
                "file_name": all_metadata[i]["file_name"],
                "chunk_id":  all_metadata[i]["chunk_id"]
            },
            vector=all_embeddings[i].tolist()
        )
weaviate_index_time = time.time() - weaviate_index_start
print(f"Weaviate indexing time: {weaviate_index_time:.2f}s\n")

# ---- Search functions ----
def qdrant_search_detailed(query_emb, k=50):
    results = client_qdrant.query_points(
        collection_name=qdrant_collection, query=query_emb, limit=k, with_payload=True
    ).points
    return [(r.payload["file_name"], r.payload.get("chunk_id")) for r in results]

def weaviate_search_detailed(query_emb, k=50):
    results = col.query.near_vector(near_vector=query_emb, limit=k, return_properties=["file_name", "chunk_id"])
    return [(r.properties["file_name"], r.properties.get("chunk_id")) for r in results.objects]

INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 19635


Re-indexing into Weaviate...


Weaviate upload: 100%|██████████| 88275/88275 [03:03<00:00, 480.33it/s]


Weaviate indexing time: 187.05s



## qdrant

In [6]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

dim = all_embeddings.shape[1]

# ---- Qdrant ----
client_qdrant = QdrantClient(":memory:")
qdrant_collection = "books_jina3"

client_qdrant.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
)

print("Re-indexing into Qdrant...")
qdrant_index_start = time.time()
for i in tqdm(range(0, len(all_embeddings), 512), desc="Qdrant upload"):
    batch_end = min(i + 512, len(all_embeddings))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings[j].tolist(),
            payload={
                "file_name": all_metadata[j]["file_name"],
                "chunk_id": all_metadata[j]["chunk_id"]
            }
        )
        for j in range(i, batch_end)
    ]
    client_qdrant.upsert(collection_name=qdrant_collection, points=points)
qdrant_index_time = time.time() - qdrant_index_start
print(f"Qdrant indexing time: {qdrant_index_time:.2f}s\n")

Re-indexing into Qdrant...


Qdrant upload:  23%|██▎       | 39/173 [00:20<01:00,  2.23it/s]/tmp/ipykernel_18374/961077417.py:30: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client_qdrant.upsert(collection_name=qdrant_collection, points=points)
Qdrant upload: 100%|██████████| 173/173 [01:36<00:00,  1.80it/s]

Qdrant indexing time: 96.06s



# evaluation

# 10%

In [8]:
benchmark_df_10 = pd.read_csv("/content/drive/MyDrive/replacement_benchmark_10pct.csv")


qdrant_robustness_10   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_10, 10)
weaviate_robustness_10 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_10, 10)

robustness_10_df = pd.DataFrame([qdrant_robustness_10, weaviate_robustness_10])
print(robustness_10_df.to_string(index=False))

robustness_10_df.to_csv("/content/drive/MyDrive/replacement_results_jina3_10pct.csv", index=False)

Evaluating Weaviate (10%): 100%|██████████| 1000/1000 [01:07<00:00, 14.83it/s]


vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 10%_token_deletion      0.866      0.985       0.987        0.856        0.979         0.981           537.39          620.91
    Weaviate jina-v5-nano 10%_token_deletion      0.869      0.987       0.988        0.853        0.979         0.980            12.53           21.65


# 20%

In [9]:
benchmark_df_20 = pd.read_csv("/content/drive/MyDrive/replacement_benchmark_20pct.csv")


qdrant_robustness_20   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_20, 20)
weaviate_robustness_20 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_20, 20)

robustness_20_df = pd.DataFrame([qdrant_robustness_20, weaviate_robustness_20])
print(robustness_20_df.to_string(index=False))

robustness_20_df.to_csv("/content/drive/MyDrive/replacement_results_jina3_20pct.csv", index=False)

Evaluating Weaviate (20%): 100%|██████████| 1000/1000 [01:05<00:00, 15.33it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 20%_token_deletion      0.852      0.976       0.983        0.831        0.965         0.969           516.37          599.23
    Weaviate jina-v5-nano 20%_token_deletion      0.843      0.973       0.980        0.824        0.959         0.961            10.31           13.63


# 30 %

In [ ]:
benchmark_df_30 = pd.read_csv("/content/drive/MyDrive/replacement_benchmark_30pct.csv")


qdrant_robustness_30   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_30, 30)
weaviate_robustness_30 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_30, 30)

robustness_30_df = pd.DataFrame([qdrant_robustness_30, weaviate_robustness_30])
print(robustness_30_df.to_string(index=False))

robustness_30_df.to_csv("/content/drive/MyDrive/replacement_results_jina3_30pct.csv", index=False)

Evaluating Weaviate (30%):  90%|████████▉ | 895/1000 [00:57<00:08, 12.00it/s]

# 40 %

In [11]:
benchmark_df_40 = pd.read_csv("/content/drive/MyDrive/replacement_benchmark_40pct.csv")


qdrant_robustness_40   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_40, 40)
weaviate_robustness_40 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_40, 40)

robustness_40_df = pd.DataFrame([qdrant_robustness_40, weaviate_robustness_40])
print(robustness_40_df.to_string(index=False))

robustness_40_df.to_csv("/content/drive/MyDrive/replacement_results_jina3_40pct.csv", index=False)

Evaluating Weaviate (40%): 100%|██████████| 1000/1000 [01:06<00:00, 15.03it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 40%_token_deletion      0.483      0.719       0.824        0.404        0.600         0.651           512.63          604.88
    Weaviate jina-v5-nano 40%_token_deletion      0.479      0.707       0.811        0.399        0.581         0.628            10.66           13.66


# 50 %

In [13]:
benchmark_df_50 = pd.read_csv("/content/drive/MyDrive/replacement_benchmark_50pct.csv")


qdrant_robustness_50   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_50, 50)
weaviate_robustness_50 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_50, 50)

robustness_50_df = pd.DataFrame([qdrant_robustness_50, weaviate_robustness_50])
print(robustness_50_df.to_string(index=False))

robustness_50_df.to_csv("/content/drive/MyDrive/replacement_results_jina3_50pct.csv", index=False)

Evaluating Weaviate (50%): 100%|██████████| 1000/1000 [01:02<00:00, 15.90it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 50%_token_deletion      0.234      0.452       0.582        0.155        0.272         0.316           517.05          613.05
    Weaviate jina-v5-nano 50%_token_deletion      0.224      0.442       0.568        0.152        0.256         0.298             9.83           12.62
